# Train model phát hiện ổ gà và gờ giảm tốc (YOLOv8n)

Chạy trên Google Colab: **Runtime → Change runtime type → T4 GPU**.

Đầu vào: file `road_merged.zip` tạo bởi `tools/merge_datasets.py` (hoặc bản export YOLOv8 từ Roboflow đã có đúng 2 lớp `pothole`, `speed_bump`).
Đầu ra: `road_best.pt` — chép vào thư mục `models/` của repo.

In [ ]:
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

## 1. Đưa dataset lên Colab
Cách 1: upload file zip (chạy ô dưới, chọn file). Cách 2: để zip trên Google Drive rồi mount Drive.

In [ ]:
from google.colab import files
uploaded = files.upload()          # chọn road_merged.zip
zip_name = next(iter(uploaded))
!rm -rf /content/data && mkdir -p /content/data && unzip -q "$zip_name" -d /content/data
!find /content/data -name data.yaml

In [ ]:
# Sửa đường dẫn này theo kết quả lệnh find ở trên
DATA_YAML = "/content/data/data.yaml"
!cat $DATA_YAML

## 2. Train
- `epochs=50` đủ cho dataset vài trăm ảnh; khi có nhiều ảnh hơn có thể tăng lên 80–100.
- Augmentation bên dưới mô phỏng ảnh ESP32-CAM: đổi màu, độ sáng, lật ngang. Ảnh mờ/nhiễu nên bật thêm khi export từ Roboflow.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data=DATA_YAML, epochs=50, imgsz=640, batch=16, patience=15,
    hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, fliplr=0.5, degrees=5, scale=0.4,
    project="/content/runs", name="road", exist_ok=True,
)

## 3. Đánh giá trên tập test
Ghi lại mAP50, precision, recall và confusion matrix cho báo cáo (gửi TV4).

In [ ]:
best = YOLO("/content/runs/road/weights/best.pt")
metrics = best.val(data=DATA_YAML, split="test", imgsz=640)
print(f"mAP50 = {metrics.box.map50:.3f}   mAP50-95 = {metrics.box.map:.3f}")
print(f"precision = {metrics.box.mp:.3f}   recall = {metrics.box.mr:.3f}")

In [ ]:
from IPython.display import Image, display
for f in ["confusion_matrix.png", "results.png", "val_batch0_pred.jpg"]:
    p = f"/content/runs/road/{f}"
    try: display(Image(filename=p, width=700))
    except Exception: print("không có", p)

## 4. Tải model về
Đổi tên thành `road_best.pt`, chép vào `models/` của repo (hoặc đặt tên khác và sửa `MODEL_PATH` trong `.env`).

In [ ]:
!cp /content/runs/road/weights/best.pt /content/road_best.pt
files.download("/content/road_best.pt")